# UPDATE, DELETE a Transakce

V předchozích noteboocích jsme se naučili data číst (`SELECT`) a vkládat (`INSERT`). Nyní se naučíme data **měnit** (`UPDATE`), **mazat** (`DELETE`) a pracovat s **transakcemi** — skupinou příkazů, které se buď provedou všechny najednou, nebo vůbec žádný.

---

## Přehled příkazů v tomto notebooku

| Příkaz | Kategorie | Popis |
|---|---|---|
| `UPDATE` | DML | Změní hodnoty existujících záznamů |
| `DELETE` | DML | Smaže záznamy z tabulky |
| `TRUNCATE` | DDL | Smaže vše a resetuje AUTO_INCREMENT |
| `START TRANSACTION` | TCL | Zahájí transakci |
| `COMMIT` | TCL | Potvrdí a uloží změny transakce |
| `ROLLBACK` | TCL | Vrátí všechny změny zpět |
| `SAVEPOINT` | TCL | Uloží mezistav uvnitř transakce |
| `ROLLBACK TO` | TCL | Vrátí se k uloženému mezistavu |

In [ ]:
# Instalace knihovny (stačí jednou)
! pip install mysql-connector-python

## Připojení a příprava dat

Vytvoříme ukázkovou tabulku `automobily` pro všechny příklady v tomto notebooku:

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL UNIQUE,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(50) NOT NULL DEFAULT 'B'
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace) VALUES
        ('1A11111', 4, 190, 3, 'B'),
        ('2B22222', 2, 220, 2, 'B'),
        ('3C33333', 5, 160, 5, 'B'),
        ('4D44444', 3, 250, 2, 'A'),
        ('5E55555', 8, 120, 10, 'D')
""")
mydb.commit()

def vypis():
    mycursor.execute("SELECT id, spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace FROM automobily ORDER BY id")
    print(f"{'id':<4} {'spz':<9} {'sedadla':<9} {'max km/h':<10} {'nosnost':<9} kvalifikace")
    print("-" * 52)
    for r in mycursor.fetchall():
        print(f"{r[0]:<4} {r[1]:<9} {r[2]:<9} {str(r[3]):<10} {str(r[4]):<9} {r[5]}")
    print()

print("✅ Data připravena:")
vypis()

---

# ⚠️ Zlaté pravidlo UPDATE a DELETE

> **Před spuštěním UPDATE nebo DELETE si vždy nejdřív spusťte odpovídající SELECT se stejným WHERE!**
>
> ```sql
> -- 1. Ověřte, co změníte:
> SELECT * FROM automobily WHERE spz = '1A11111';
>
> -- 2. Teprve pak spusťte:
> UPDATE automobily SET max_rychlost = 200 WHERE spz = '1A11111';
> ```
>
> UPDATE nebo DELETE **bez WHERE** ovlivní **VŠECHNY záznamy** v tabulce — bez výjimky!

---

# UPDATE — změna existujících dat

## Syntaxe

```sql
UPDATE nazev_tabulky
SET sloupec_1 = nova_hodnota,
    sloupec_2 = nova_hodnota
WHERE podminka;
```

### Pravidla
- `WHERE` **určuje, které záznamy se změní** — bez něj se změní všechny!
- Lze měnit **libovolný počet sloupců** najednou
- Po `UPDATE` je nutné zavolat `mydb.commit()`

In [ ]:
# Příklad 1: Změna jednoho sloupce pro jeden záznam
mycursor.execute("""
    UPDATE automobily
    SET max_rychlost = 210
    WHERE spz = '1A11111'
""")
mydb.commit()
print(f"Změněno: {mycursor.rowcount} záznam(ů)")
vypis()

## UPDATE — více sloupců najednou

In [ ]:
# Příklad 2: Změna více sloupců pro jeden záznam
mycursor.execute("""
    UPDATE automobily
    SET pocet_sedadel = 6,
        nosnost = 4
    WHERE id = 3
""")
mydb.commit()
print(f"Změněno: {mycursor.rowcount} záznam(ů)")
vypis()

## UPDATE — s výpočtem z aktuální hodnoty

Nová hodnota může být **vypočtena z aktuální** hodnoty sloupce:

In [ ]:
# Příklad 3: Snížení max. rychlosti všech aut s kvalifikací 'B' o 10 km/h
mycursor.execute("""
    UPDATE automobily
    SET max_rychlost = max_rychlost - 10
    WHERE nutna_kvalifikace = 'B'
""")
mydb.commit()
print(f"Změněno: {mycursor.rowcount} záznam(ů)")
vypis()

## Cvičení 1 — UPDATE

1. Změňte počet sedadel automobilu se SPZ `'4D44444'` na **4**.
2. Všem automobilům s `max_rychlost` nižší než **170 km/h** zvyšte nosnost o **2**.
3. Zobrazte tabulku a ověřte výsledek.

<details>
<summary>🔎 Ukázka řešení (klikni pro zobrazení)</summary>

```python
mycursor.execute("UPDATE automobily SET pocet_sedadel = 4 WHERE spz = '4D44444'")
mycursor.execute("UPDATE automobily SET nosnost = nosnost + 2 WHERE max_rychlost < 170")
mydb.commit()
vypis()
```

</details>

In [ ]:
# Cvičení 1: Váš kód sem 👇


---

# DELETE — mazání záznamů

## Syntaxe

```sql
DELETE FROM nazev_tabulky
WHERE podminka;
```

### Vlastnosti
- Smaže záznamy **splňující podmínku**
- **Bez WHERE smaže VŠECHNY záznamy** v tabulce
- Zachovává strukturu tabulky i hodnotu `AUTO_INCREMENT` čítače
- Lze vrátit pomocí `ROLLBACK` (pokud používáme transakce)

In [ ]:
# Příklad: Smazání jednoho záznamu
mycursor.execute("DELETE FROM automobily WHERE spz = '5E55555'")
mydb.commit()
print(f"Smazáno: {mycursor.rowcount} záznam(ů)")
vypis()

## TRUNCATE vs DELETE

| Vlastnost | `DELETE FROM tabulka` | `TRUNCATE TABLE tabulka` |
|---|---|---|
| Maže | Záznamy dle WHERE (bez WHERE = vše) | Vždy všechny záznamy |
| AUTO_INCREMENT | **Neresetu**je čítač | **Resetuje** čítač na 1 |
| Lze ROLLBACK | ✅ Ano | ❌ Ne (DDL příkaz) |
| Rychlost | Pomalejší | Rychlejší |

> `TRUNCATE` použijte pro **úplné vymazání a reset** tabulky.

In [ ]:
# Ukázka rozdílu v AUTO_INCREMENT
mycursor.execute("INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace) VALUES ('5E55555', 8, 120, 10, 'D')")
mydb.commit()
mycursor.execute("SELECT id FROM automobily ORDER BY id DESC LIMIT 1")
print("Nejvyšší id před TRUNCATE:", mycursor.fetchone()[0])

mycursor.execute("TRUNCATE TABLE automobily")
mydb.commit()

# Vložíme nový záznam — čítač začíná od 1
mycursor.execute("INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace) VALUES ('1A11111', 4, 190, 3, 'B')")
mydb.commit()
mycursor.execute("SELECT id FROM automobily ORDER BY id DESC LIMIT 1")
print("id po TRUNCATE + INSERT:", mycursor.fetchone()[0])  # → 1

## Cvičení 2 — DELETE

1. Obnovte data — spusťte znovu buňku „Připojení a příprava dat".
2. Smažte všechny automobily s `max_rychlost > 200`.
3. Smažte automobil s `id = 2`.
4. Zobrazte výsledek.

<details>
<summary>🔎 Ukázka řešení (klikni pro zobrazení)</summary>

```python
mycursor.execute("DELETE FROM automobily WHERE max_rychlost > 200")
mycursor.execute("DELETE FROM automobily WHERE id = 2")
mydb.commit()
vypis()
```

</details>

In [ ]:
# Cvičení 2: Váš kód sem 👇


---

# Transakce (Transactions)

**Transakce** je skupina SQL příkazů, které tvoří **jeden nedělitelný celek** — buď se provedou **všechny**, nebo se neprovede **žádný**.

## Proč transakce?

Příklad: bankovní převod

```
1. UPDATE ucty SET zustatek = zustatek - 1000 WHERE id = 1   ← odečteme odesílateli
2. UPDATE ucty SET zustatek = zustatek + 1000 WHERE id = 2   ← přičteme příjemci
```

Co kdyby program havaroval po kroku 1? Peníze by zmizely! Transakce zajistí, že proběhnou oba kroky, nebo žádný.

## ACID vlastnosti

| Vlastnost | Popis |
|---|---|
| **A**tomičnost | Vše nebo nic |
| **C**onzistentnost | Data zůstanou v platném stavu |
| **I**zolace | Transakce se navzájem neovlivňují |
| **T**rvanlivost | Potvrzené změny přežijí výpadek |

## Příkazy TCL

| Příkaz | Popis |
|---|---|
| `START TRANSACTION` nebo `BEGIN` | Zahájí transakci |
| `COMMIT` | Potvrdí změny — uloží je trvale |
| `ROLLBACK` | Zruší všechny změny od posledního `START TRANSACTION` |
| `SAVEPOINT jmeno` | Uloží pojmenovaný mezistav uvnitř transakce |
| `ROLLBACK TO jmeno` | Vrátí se k danému mezistavu (zbytek transakce pokračuje) |

> **mysql.connector** má výchozí `autocommit = False` — každý příkaz je automaticky v transakci a musíte volat `mydb.commit()` nebo `mydb.rollback()`.

## Obnovení dat pro příklady s transakcemi

In [ ]:
# Obnovení dat
mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL UNIQUE,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(50) NOT NULL DEFAULT 'B'
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace) VALUES
        ('1A11111', 4, 190, 3, 'B'),
        ('2B22222', 2, 220, 2, 'B'),
        ('3C33333', 5, 160, 5, 'B'),
        ('4D44444', 3, 250, 2, 'A'),
        ('5E55555', 8, 120, 10, 'D')
""")
mydb.commit()
print("✅ Data obnovena.")
vypis()

## COMMIT — potvrzení transakce

In [ ]:
mycursor.execute("START TRANSACTION")

mycursor.execute("UPDATE automobily SET max_rychlost = 200 WHERE spz = '1A11111'")
mycursor.execute("UPDATE automobily SET pocet_sedadel = 3 WHERE spz = '2B22222'")

mydb.commit()
print("✅ COMMIT — obě změny uloženy.")
vypis()

## ROLLBACK — vrácení změn

In [ ]:
print("Stav před transakcí:")
vypis()

mycursor.execute("START TRANSACTION")
mycursor.execute("DELETE FROM automobily WHERE nutna_kvalifikace = 'B'")

mycursor.execute("SELECT COUNT(*) FROM automobily")
print(f"V transakci (ještě nepotvrzeno) — zbývá: {mycursor.fetchone()[0]} záznamů")

mydb.rollback()
print("\n↩️  ROLLBACK — DELETE zrušen.")
print("Stav po ROLLBACK:")
vypis()

## SAVEPOINT — mezibod v transakci

`SAVEPOINT` vytvoří **pojmenovaný bod zvratu** uvnitř transakce.
`ROLLBACK TO` vrátí změny jen k tomuto bodu — zbytek transakce může pokračovat.

In [ ]:
mycursor.execute("START TRANSACTION")

# Krok 1 — UPDATE (chceme zachovat)
mycursor.execute("UPDATE automobily SET max_rychlost = 999 WHERE id = 1")
mycursor.execute("SAVEPOINT po_update")
print("💾 SAVEPOINT 'po_update' uložen.")

# Krok 2 — DELETE (nechceme zachovat)
mycursor.execute("DELETE FROM automobily WHERE id = 2")
print("⚠️  DELETE id=2 proveden (ale ještě nepotvrzeno).")

mycursor.execute("ROLLBACK TO po_update")
print("↩️  ROLLBACK TO 'po_update' — DELETE zrušen, UPDATE zůstal.")

mydb.commit()
print("✅ COMMIT.")
vypis()

## try/except + rollback — doporučený vzor v Pythonu

V reálném kódu zachycujeme chyby pomocí `try/except` a při chybě automaticky voláme `ROLLBACK`:

```python
try:
    mycursor.execute("START TRANSACTION")
    # ... UPDATE / DELETE ...
    mydb.commit()
    print("✅ Úspěch")
except Exception as e:
    mydb.rollback()
    print(f"❌ Chyba, změny vráceny: {e}")
finally:
    mydb.close()
```

In [ ]:
def atomicky_prevod(id_z, id_na, zmena_rychlosti):
    """Atomicky přenese část nosnosti a změní rychlost — obě změny nebo žádná."""
    try:
        mycursor.execute("START TRANSACTION")
        mycursor.execute(f"UPDATE automobily SET nosnost = nosnost - 1 WHERE id = {id_z}")
        mycursor.execute(f"UPDATE automobily SET nosnost = nosnost + 1, max_rychlost = max_rychlost + {zmena_rychlosti} WHERE id = {id_na}")
        mydb.commit()
        print(f"✅ Převod mezi id={id_z} a id={id_na} úspěšný.")
    except Exception as e:
        mydb.rollback()
        print(f"❌ Chyba, transakce zrušena: {e}")

atomicky_prevod(1, 3, 5)
vypis()

# Záměrná chyba (neexistující sloupec)
try:
    mycursor.execute("START TRANSACTION")
    mycursor.execute("UPDATE automobily SET neexistujici = 0 WHERE id = 1")
    mydb.commit()
except Exception as e:
    mydb.rollback()
    print(f"❌ Zachycena chyba: {e}")

## Cvičení 3 — Transakce se SAVEPOINT

1. Spusťte transakci.
2. Vložte nový automobil (`spz='6F66666'`, 5 sedadel, rychlost 170, nosnost 3, kvalifikace `'B'`).
3. Upravte nosnost `id=1` na **8**.
4. Uložte `SAVEPOINT pred_delete`.
5. Smažte automobil `id=3`.
6. Proveďte `ROLLBACK TO pred_delete`.
7. Proveďte `COMMIT`.
8. Ověřte: INSERT a UPDATE přežily, DELETE byl zrušen.

<details>
<summary>🔎 Ukázka řešení (klikni pro zobrazení)</summary>

```python
mycursor.execute("START TRANSACTION")
mycursor.execute("INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace) VALUES ('6F66666', 5, 170, 3, 'B')")
mycursor.execute("UPDATE automobily SET nosnost = 8 WHERE id = 1")
mycursor.execute("SAVEPOINT pred_delete")
mycursor.execute("DELETE FROM automobily WHERE id = 3")
mycursor.execute("ROLLBACK TO pred_delete")
mydb.commit()
vypis()
```

</details>

In [ ]:
# Cvičení 3: Váš kód sem 👇


## Cvičení 4 — Kompletní úloha

Simulujte jednoduchý skladový systém s ochranou dat:

1. Vytvořte tabulku `sklad` (id INT PK AUTO_INCREMENT, nazev VARCHAR(50) NOT NULL, mnozstvi INT NOT NULL DEFAULT 0, cena_kus DECIMAL(10,2) NOT NULL).
2. Vložte alespoň 4 produkty s různými cenami a množstvím.
3. V transakci: snižte množství prvního produktu o 5 (výdej), zvyšte druhému o 10 (příjem). Potvrďte.
4. Zkuste nastavit záporné množství (neplatné) — zachyťte výjimku a proveďte `ROLLBACK`.
5. Zobrazte finální stav skladu.
6. Smažte tabulku a odpojte se.

<details>
<summary>🔎 Očekávaný průběh</summary>

```
✅ Výdej a příjem proběhly.
❌ Záporné množství není povoleno — transakce zrušena.
--- Sklad ---
id=1  ...  množství: 45
id=2  ...  množství: 110
...
```

</details>

In [ ]:
# Cvičení 4: Váš kód sem 👇

# Nezapomeňte na konci:
# mycursor.execute("DROP TABLE IF EXISTS sklad")
# mydb.commit()
# mydb.close()


---

## 📋 Shrnutí

| Příkaz | Popis | Pozor |
|---|---|---|
| `UPDATE tabulka SET ... WHERE ...` | Změní hodnoty | Vždy použij WHERE! |
| `DELETE FROM tabulka WHERE ...` | Smaže záznamy | Vždy použij WHERE! |
| `TRUNCATE TABLE tabulka` | Smaže vše + reset čítače | Nelze ROLLBACK |
| `START TRANSACTION` | Zahájí transakci | — |
| `COMMIT` | Uloží změny trvale | — |
| `ROLLBACK` | Zruší vše od START TRANSACTION | — |
| `SAVEPOINT jmeno` | Uloží mezibod | — |
| `ROLLBACK TO jmeno` | Vrátí se k mezibodu | — |

### Doporučený vzor
```python
try:
    mycursor.execute("START TRANSACTION")
    # ... DML příkazy ...
    mydb.commit()
except Exception as e:
    mydb.rollback()
    print(f"Chyba: {e}")
finally:
    mydb.close()
```